# Latent Dirichlet Allocation

* Prima di iniziare con la descrizione del progetto, ho esposto una sintesi di alcuni richiami teorici per contestualizzare l'oggetto di questo studio.

  I contenuti che seguono sono stati realizzati mettendo insieme [questa fonte](https://thesis.unipd.it/retrieve/8e03daa1-bdf5-4739-934a-d31420a6ffcb/Analisi%20ed%20applicazione%20di%20un%20algoritmo%20di%20Topic%20Modeling.pdf), appunti presi a lezione e le mie conoscenze pregresse in ambito probabilistico.



Il LDA (Latent Dirichlet Allocation) è un modello **generativo**, cioè un modello che descrive e **semplifica** il **processo di creazione di un documento testuale** all'interno di un corpus , assumendo che esso avvenga secondo una serie di passaggi probabilistici.
Nello specifico, il LDA adotta un **approccio bayesiano** : costruisce un processo stocastico che si presume abbia generato i  vari testi osservati nel corpus a partire da specifiche **ipotesi a priori**.
Il termine "**latente**" si riferisce a quelle variabili nascoste che intervengono nel processo di generazione del testo e che non sono direttamente osservabili. Nel contesto del LDA, tali variabili latenti corrispondono ai **topic**, intesi come distribuzioni di probabilità sull'insieme delle parole utilizzate nei vari documenti.

## Approccio bayesiano
### Regola di Bayes
 $$ \mathbb{P}(\theta \mid X )=\frac{\mathbb{P}(X \mid\theta)P(\theta)}{\mathbb{P}(X)} $$
 dove:
 * $ \mathbb{P}(\theta \mid X )$ è la Posterior distribution e rappresenta la probabilità a posteriori (dopo aver preso in considerazione i dati osservati) di $\theta$
 * $\mathbb{P}(\theta)$ è la prior distribution e definisce le ipotesi iniziali sul parametro $\theta$
 * $\mathbb{P}(X \mid \theta)$ è la cosidetta verosimiglianza e rappresenta la probabilità di
 osservare i dati assumendo il parametro $\theta$
 * $\mathbb{P}(X)$ è la probabilità di X che si può esprimere, condizionando su tutti i possibili valori che può assumere $\theta$, con la seguente formula:
 $$\mathbb{P}(X)=\int \mathbb{P}(X \mid \theta)\mathbb{P}(\theta) d\theta $$

 Quest' ultimo integrale, è spesso difficile da computare a causa dell'elevata dimensionalità dello spazio dei parametri. Per ovviare a questo problema, si utilizzano tecniche di stima che permettono, con buona approssimazione, di ottenere ugualmente un' espressione per la posterior distribution

## Descrizione del processo generativo
Descriviamo di seguito come avviene il processo generativo nel modello LDA, scegliendo un numero fissato T di topic, un numero D di documenti nel corpus, e un numero N di parole del vocabolario
1. Per ogni topic $t$ $\in \{1 \dots \text{T}\}$:
+ (a) A partire da una distr. di Dirichlet simmetrica, si estrae il parametro di una distr. multinomiale del topic $t$ sulle le parole del vocabolario, $\phi_z \sim  \text{Dir}(\beta)$
2. Per ogni documento $d$ $\in \{1 \dots D \}$
+ (a) A partitre da una distr. di Dirichlet simmetrica si estrae il parametro di una distr. multinomiale del documento $d$ sui topic, $\theta_d \sim \text{Dir}(\alpha)$
+ (b) Per ogni parola $w$ $\in \{ 1 \dots N_d\}$
   * (i) Viene estratto un topic dalla distr. del documento $d$ sui topic, $z_{dn} \sim \text{Multinomial} (\theta_d)$
   * (ii) Viene estratta una parola dalla distr. dei topic sulle parole, $w_{dn} \sim \text{Multinomial}(\phi_z)$
   ### Osservazione
   Nel caso del modello LDA, la prior distribution è dunque la distribuzione di Dirichlet che viene usata per estarre i parametri delle distribuzioni multinomiali:      
     * $\theta_d=(\theta_{d,1} \dots \theta_{d,T} )$ è la distr. dei documenti sui topic,e ogni entrata $i$ rappresenta la prob. del topic $i$ per il documento $d$ per cui $\sum_{i=1}^T \theta_{d,i}=1$
     * $\phi_z=(\phi_{z,1} \dots \phi_{z,N} )$ è la dist. dei topic sulle parole, e ogni entrata $j$ rappresenta la prob. della $j$-esia parola per il topic $t$ per cui $\sum_{j=1}^N \phi_{z,j}=1$

     ### Probabilità congiunta
     Riportiamo di seguito l' espressione per la probabilità congiunta del modello LDA, valida sotto le ipotesi di **cambiabilità dei documenti** e **scambiabilità delle parole**

  $$p(\textbf{z}, \textbf{w}, \theta_{1:D}, \phi_{1:T} \mid \alpha, \beta)= \prod_{t=1}^T p(\phi_t \mid \beta) \prod_{d=1}^D p(\theta_d \mid \alpha) \prod _{n=1}^{N_d}p(z_{dn}\mid \theta_d)p(w_{dn} \mid z_{dn}, \phi_{1:T})$$
  dove :   
  * $z_{dn}$ rappresenta l'assegnazione di un certo topic all' $n$-esima parola del documento d
  * $\textbf{z}$ è l'assegnazione di topic per tutte le parole
  * $\textbf{w}$ è l'insieme di tutte le parole che compaiono nel corpus di documenti

# Topic Modelling
La classe degli algoritmi di topic modelling, è quella classe di algoritmi che si occupa di ricavare informazioni da un insieme molto vasto di documenti, fornendo misure quantitative che possono essere utilizzate per identificare il contenuto dei testi, ossia definire quali sono gli argomenti trattati (topic).
 Il topic modelling, è una tecnica non supervisionata.Questo significa che non effettua l'apprendimento a partire da etichette preassegnate a ciascun documento della collezione, bensì ricava le distribuzioni di probabilità sulle parole del vocabolario associate a ciascuno dei topic. Attraverso l'analisi delle parole più probabili, sta a noi risalire a quale sia l'argomento trattato.

## Topic modelling tramite LDA
L'obiettivo centrale del modello LDA è rappresentato dall' inversione del processo generativo, al fine di determinare le variabili latenti date le variabili osservate (sequenza delle parole nei vari testi).
Le variabili latenti da determinare sono:
* $z_{dn} →$ assegnazione dei topic a ciascuna parola nei dcoumenti. Per ogni parola di un documento, vogliamo determinare a quale topic è assegnata
* $\theta_{1:D} →$ la distribuzione sui topic associata ad ogni documento $1 \dots D$
* $\phi_{1:T} →$ la distribuzione sulle parole associata ad ogni topic $1 \dots T$.

Il problema inferenziale che deve essere risolto, consiste nel calcolo della distr. a posteriori :
$$p(\textbf{z}, \theta_{1:D}, \phi_{1:T} \mid \textbf{w}, \alpha, \beta) = \frac{p(\textbf{z}, \theta_{1:D}, \phi_{1:T} , \textbf{w}\mid \alpha, \beta)}{p(\textbf{w} \mid \alpha,\beta)} $$
con la formula che è l'applicazione della formula di Bayes su più eventi:
$$\mathbb{P}(A \mid B \cap C)=\frac{ \mathbb{P}(A \cap B \cap C)} {\mathbb{P}(B \cap C)}=\frac{\mathbb{P}(A \cap B \mid C) \mathbb{P}(C)}{\mathbb{P}(B \cap C)}=\frac{\mathbb{P}(A \cap B \mid C )}{\mathbb{P}(B \mid C)} $$
Questa distribuzione, tuttavia, non è trattabile analiticamnete, essendo il denominatore ottenibile solo mediante integrazione da effettuare su uno spazio parametrico di elevata dimensionalità.
Quindi l'inferenza esatta è intrattabile, ma si può utilizzare l'inferenza approssimata utilizzando ad esempio il metodo del **Gibbs Sampling**. L'obiettivo consiste dunque nell'approssimare la distribuzione a posteriori ed ottenere una stima dei parametri tramite ottimizzazione.

# Gibbs Sampling
Il Gibbs Sampling, fa parte della classe di algoritmi del framework Markov Chain Montecarlo (MCMC) e viene usato per il **Topic Modelling**  nel contesto del LDA.
## Cenni su catene di Markov e metodo Montecarlo
### Catene di Markov
Una catena di Markov, è un processo stocastico costituito da una sequenza di variabili aleatorie $(X_n)_n$ che assumono valori in un certo spazio degli stati e che evolve nel tempo in accordo alla **proprietà di Markov** ovvero:
per ogni istante di tempo $n$, e per ogni possibile scelta di una sequenza di stati $x_0,x_1,\dots x_n$ che soddisfa $\mathbb{P}(X_0=x_0,\dots X_n=x_n)>0$ vale che $\mathbb{P}(X_{n+1}=x_{n+1} \mid X_n=x_n,\dots X_0=x_0)=\mathbb{P}(X_{n+1}=x_{n+1} \mid X_n=x_n)=P(x_n,x_{n+1})$.
 In altre parole, il processo, all’istante $n$, sceglie la sua posizione
 successiva tenedo conto solamente della sua posizione attuale $X_n$,
 dimenticando tutte le mosse effettuate in passato.
 Le catene di maggiore interesse, sono quelle che ammettono una distribuzione di probabilità stazionaria, ossia una distr. $\pi$ che soddisfa $\pi P=\pi $, vale a dire $\forall x$ $\pi(x)=\sum_y \pi(y)P(y,x)$.  Possiamo interpretare la definizione nella maniera seguente:
 se la distribuzione attuale è $\pi$, effettuando una transizione della
 catena, rimarremo distribuiti nello stesso modo.
 Queste distribuzioni assumono un ruolo centrale nello studio
 delle catene di Markov, poiché, sotto opportune condizioni, la
 misura stazionaria coincide con la distribuzione asintotica della
 catena.
 ### Metodo Montecarlo e relazione con la statistica Bayesiana
 Gli algoritmi MCMC, sfruttano le proprietà delle catene di Markov con l'obiettivo di simulare il campionamento da una certa distribuzione di probabilità target $\pi$ o analogamente computare $\mathbb{E}_{\pi}(f)$, il valore atteso di una funzione sotto la distribuzione $\pi$. Infatti, l'idea su cui si basano tali algoritmi, consiste nel costruire una catena di Markov che converga a $\pi$ (quindi che abbia $\pi$ come distr. stazionaria). La ragione principale alla base dell'implementazione di questo tipo  di tecniche, risiede nel fatto che le distribuzioni target assumono spesso espressioni  complicate, che dipendono da costanti di normalizzazione molto difficili da computare.
 Gli algoritmi MCMC trovano una naturale applicazione nel contesto dell' inferenza bayesiana. In questo tipo di paradigma infatti, come abbiamo visto, l'interesse è volto ad ottenere la densità **a posteriori** del vettore di parametri $\theta$.

# Utlizzo del Gibbs Sampling in LDA
L'obiettivo del Gibbs sampling applicato al modello LDA, è quello di costruire una catena di Markov che abbia la posterior distribution $p(\textbf{z}, \theta_{1:D}, \phi_{1:T} \mid \textbf{w}, \alpha, \beta)$ come distr. stazionaria, in modo tale che il comportamento della catena dopo un grande numero di passi approssimi bene la distr. stazionaria (in virtu del Teorema Ergodico).
Ogni stato della catena è rappresentato dalle variabili $z_{dn}$ che rappresentano le assegnazioni dei topi per ciascuna parola. Le transizioni tra gli stati successivi, invece, avvegono tramite il campionamento sequenziale delle assegnazioni $z_{dn}$ da una distr. condizionale $$ P(z_{dn}=k \mid z_{-dn}, w_{dn}, d)$$
dove :
* $z_{dn}$ corrisponde al topic campionato *k* assegnato alla parola $w_{dn}$
* $z_{-dn}$ rappresenta le  assegnazioni del topic in questione a tutte le altre parole.
Le variabili campionate vengono dunque modificate sequenzialmente fino a quando non viene raggiunta la distr. stazionaria della catena. Dopo aver stimato la distr. a posteriori di  $z_{dn}$, questa  viene sfruttata per dedurre le distr. $\phi_{1:T}$ e $\theta_{1:D}$.
La distr. condizionale del Gibbs Sampling è definita come:
$$ P(z_{dn}=k \mid z_{-dn}, w_{dn}, d) \propto \frac{C_{w_i,k}^1+\beta}{\sum_w C_{w,k}^1+ N\beta} \frac{C_{d,k}^2+\alpha}{\sum_t C_{d,t}^2 +T\alpha}$$
dove:
* $C^1$ è è la matrice il cui elemento $w_i,k$ rappresenta il conteggio di quante volte la parola $w_i$ è stata assegnata al topic $k$
* $C^2$ è è la matrice il cui elemento $d,k$ rappresenta il numero di volte che l'argomento $k$ è stato assegnato ad una qualsiasi parola del documento $d$
* $\sum_t C_{d,t}^2$ è il conteggio totale di assegnazioni di topic alle parole nel documento $d$
* $\sum_w C_{w,k}^1$ è il conteggio totale di parole assegnate al topic $k$.

Quindi per semplificare possiamo dire che questa espressione tiene conto di :
* Quanta "presenza di topic $k$" c'è nel documento d
* Quanto la parola $w$ partecipa al topic $k$

# Esecuzione dell' algoritmo
Per ogni documento $\{1 \dots D \}$ del corpus ripeti :

1. Si assegna a ciascuna parola $w_{dn}$ del documento un topic casuale $\{1 \dots T \}$
2. Per ciascuna parola $w_{dn}$ si iterano i seguenti passaggi:
    * Viene tolta l' assegnazione corrente della parola $w_{dn}$ all' argomento $k$.  Conseguentemente gli elementi $C_{w_i,k}^1$ e $C_{d,k}^2$ vengono decrementati di uno.
    * Un nuovo argomento viene campionato dalla distr. condizionale del Gibbs Sampling  ed assegnato alla parola $w_{dn}$: le matrici vengono aggiornate di conseguenza

Dunque, ad ogni iterazione,  si passa attraverso ciascun termine del corpus e si aggiorna l'assegnazione dei topic utilizzando una distr. condizionale basata sulle assegnazioni attuali per tutte le altre parole ($z_{-dn}$).

Al termine di un passaggio completo su tutte le parrole, si ottiene un campione di Gibbs che rappresenta lo stato corrente delle assegnazioni $z_{dn}$ per tutto il corpus di documenti. Questo processo viene ripetuto molte volte: durante la fase iniziale, detta **burn-in**, i campioni vengono scartati perchè non rappresentano accuratamente la distr. a posteriori, poi i campioni successivi, che iniziano a convergere verso la distr. target, vengono salvati a intervalli regolari per ridurre le correlazioni tra campioni consecutivi.
### Stima di $\theta_{1:D}$ e $\phi_{1:T}$
Per valutare il contenuto tematico dei documenti si richiedono le stime di $\theta_{1:D}$ e $\phi_{1:T}$. Queste grandezze possono essere facilmente ottenute a partire dai rapporti definiti in precedenza:

* $$\theta_{d,k}=\frac{C_{d,k}^2+\alpha}{\sum_t C_{d,t}^2+T\alpha} $$

* $$\phi_{k,wi}=\frac{C_{k,w_i}^1+\beta}{\sum_w C_{k,w}^1+N\beta} $$


# Descrizione del lavoro
In questo progetto, mi occuperò di implementare l'algoritmo di topic modelling utilizzando il modello LDA, a partire da un dataset contenente recensioni pubblicate sul portale Tripadvisor da utenti italiani.

[link al dataset scaricabile da Kaggle](https://www.kaggle.com/datasets/alessandrolobello/italian-tripadvisor)

Il lavoro verrà commentato passo passo e verranno iserite le considerazioni operative maturate durante lo svolgimento.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")


Matplotlib is building the font cache; this may take a moment.


In [140]:
#import os
#current_folder=os.getcwd()
#print(current_folder)

In [11]:
#caricamento del dataset
df=pd.read_csv("archive/LLM_DF.csv")
df

,Unnamed: 0,comment,title,date,name,stars
0,0,Questo è stato un interessante e piena di sera...,Zona del canal grande,26 ottobre 2016,gfdmmad,5.0
1,1,Ottimo e ben organizzato. Nessuna coda all'ent...,Visita di piacere,26 aprile 2019,Diabolik-Eva-014,5.0
2,2,Ci siamo stati molto spesso con gli amici o da...,Da vedere a Milano,18 novembre 2018,O5262KGchrisd,5.0
3,3,Consiglio di visitare il Duomo di Milano ....e...,Il Duomo sempre affascinante,7 agosto 2019,Babellina,5.0
4,4,La vista del duomo all'uscita della metro ti t...,Eccezionale,9 dicembre 2019,Nik804,5.0
...,...,...,...,...,...,...
20053,20053,Un bel parco in mezzo ad una delle città più g...,È bellissimo!,3 ottobre 2015,Giuseppe O,5.0
20054,20054,"Abbiamo passeggiato, bellissimi i riflessi del...",Navigli,1 gennaio 2020,michy77454,5.0
20055,20055,Che dire? Non manca proprio nulla: da Beato An...,Uno dei musei più belli del mondo,18 ottobre 2017,Ninotrapani,5.0
20056,20056,Spettacolare Porta simbolo della città di Berl...,Spettacolare Porta simbolo della città di Berl...,23 settembre 2019,Vincenzo P,5.0


* Possiamo osservare che il dataset contiene più di 20000 recensioni registarte sul sito Tripadvisor
* Per ogni recensione sono riportati:
   * Identificativo progressivo
   * Titolo
   * Testo della recensione
   * data
   * nickname del recensore
   * rating(da 1 a 5 stelle)

* Otteniamo alcune informazioni sul dataset:
Vediamo quante recensioni sono presenti per ogni classe di rating (da 1 a 5 stelle)

In [12]:
df.groupby("stars").count()

,Unnamed: 0,comment,title,date,name
stars,,,,,
1.0,126,126,126,126,126
2.0,222,222,222,222,222
3.0,911,911,911,911,911
4.0,4306,4306,4306,4306,4306
5.0,14493,14493,14493,14493,14493


* Vediamo qualche info sulle date

In [13]:
print(df["date"].max())
print(df["date"].min())

 9 settembre 2021
 1 agosto 2012


In [14]:
df["date"].describe()

count                20058
unique                2990
top        15 gennaio 2019
freq                    50
Name: date, dtype: object

* Cominciamo ad importare le librerie per il processamento del testo

In [16]:
import nltk
#from nltk.tokenize import word_tokenize, RegexpTokenizer

In [18]:
#nltk.download('punkt_tab')
#print("Download effettuato correttamente")

* Prendiamo una parte di testo a titolo di esempio, concatenando 3 recensioni e separandole tra loro andando a capo

In [20]:
pezzetto=df["comment"].iloc[10:12].str.cat(sep="\n")
print(pezzetto)

Visitare Milano nel periodo di Natale e' molto suggestivo e le mille luminarie che impreziosiscono la citta'la rendono ancora pu' magica.Tra i tanti eventi cui e' possibile assistere in questo periodo, consiglio a tutti le fontane danzanti sui Navigli.Ogni sera dopo il 7 dicembre dalle 18.30 in poi sulla Darsena si assiste a uno spettacolo meraviglioso: fontane d' acqua illuminate che passano dal bianco all'azzurro ai toni piu' accessi del rosso e dell' arancio  danzando al ritmo di musiche natalazie.Nei dintorni,le tipiche bancherelle con chioschi food e drink che ti riscalderanno.Per chi lo desidera e' possibile ogni giorno dal 16 novembre al 30 dicembre 2019 fare una navigazione sui Navigli accompagnati da un pianista compositore con la partecipazione di artisti del coro della Scala.Botteghini x acquisti biglietti in loco.
Bellissimo in questa stagione , abbiamo girato in bici e a piedi. Preso la barchetta a remi nel laghetto, mangiato in uno dei tanti ristorantino sparsi e poi rila

* Convertiamo la colonna delle recensioni in formato stringa per ottenere il corpus, in cui ogni documento è una singola recensione utente.

In [21]:
corpus=df["comment"].astype(str)
print(corpus)

0        Questo è stato un interessante e piena di sera...
1        Ottimo e ben organizzato. Nessuna coda all'ent...
2        Ci siamo stati molto spesso con gli amici o da...
3        Consiglio di visitare il Duomo di Milano ....e...
4        La vista del duomo all'uscita della metro ti t...
                               ...                        
20053    Un bel parco in mezzo ad una delle città più g...
20054    Abbiamo passeggiato, bellissimi i riflessi del...
20055    Che dire? Non manca proprio nulla: da Beato An...
20056    Spettacolare Porta simbolo della città di Berl...
20057    La darsena con le varie biforcazioni così come...
Name: comment, Length: 20058, dtype: object


* Vediamo il sesto documento

In [22]:
print(corpus[5])

Vivo a Milano da una vita ma non avevo mai trovato il tempo di visitare questa chiesa nel cuore di Milano e a due passi dall'Università Cattolica.Finalmente sono riuscito nell'intento e la bellezza degli affreschi presenti al suo interno ha superato le mie aspettative!


* Insatlliamo le librerie per il processamento del testo e per l'implementazione degli algoritmi di topic modelling

In [23]:
!pip install gensim pyLDAvis spacy scikit-learn

   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   --- ------------------------------------ 2.1/24.4 MB 10.2 MB/s eta 0:00:03
   ----- ---------------------------------- 3.1/24.4 MB 8.9 MB/s eta 0:00:03
   -------- ------------------------------- 5.2/24.4 MB 8.5 MB/s eta 0:00:03
   ------------ --------------------------- 7.3/24.4 MB 8.9 MB/s eta 0:00:02
   -------------- ------------------------- 8.7/24.4 MB 8.2 MB/s eta 0:00:02
   ----------------- ---------------------- 10.5/24.4 MB 8.8 MB/s eta 0:00:02
   -------------------- ------------------- 12.6/24.4 MB 8.7 MB/s eta 0:00:02
   ---------------------- ----------------- 13.9/24.4 MB 8.3 MB/s eta 0:00:02
   ------------------------- -------------- 15.7/24.4 MB 8.7 MB/s eta 0:00:01
   ----------------------------- ---------- 17.8/24.4 MB 8.7 MB/s eta 0:00:01
   -------------------------------- ------- 19.9/24.4 MB 8.7 MB/s eta 0:00:01
   ---------------------------------- ----- 21.0/24.4 MB 8.7 MB/s eta 0:00:0

In [24]:
import string
import spacy
import gensim
from gensim import corpora
from gensim.models import CoherenceModel
import pyLDAvis.gensim_models as gensimvis
import pyLDAvis
from nltk.corpus import stopwords

nltk.download('wordnet')
nltk.download('stopwords')


from nltk.corpus import stopwords
italian_stopwords = stopwords.words('italian')


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\tmart\AppData\Roaming\nltk_data...
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\tmart\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [25]:
print(italian_stopwords)

['ad', 'al', 'allo', 'ai', 'agli', 'all', 'agl', 'alla', 'alle', 'con', 'col', 'coi', 'da', 'dal', 'dallo', 'dai', 'dagli', 'dall', 'dagl', 'dalla', 'dalle', 'di', 'del', 'dello', 'dei', 'degli', 'dell', 'degl', 'della', 'delle', 'in', 'nel', 'nello', 'nei', 'negli', 'nell', 'negl', 'nella', 'nelle', 'su', 'sul', 'sullo', 'sui', 'sugli', 'sull', 'sugl', 'sulla', 'sulle', 'per', 'tra', 'contro', 'io', 'tu', 'lui', 'lei', 'noi', 'voi', 'loro', 'mio', 'mia', 'miei', 'mie', 'tuo', 'tua', 'tuoi', 'tue', 'suo', 'sua', 'suoi', 'sue', 'nostro', 'nostra', 'nostri', 'nostre', 'vostro', 'vostra', 'vostri', 'vostre', 'mi', 'ti', 'ci', 'vi', 'lo', 'la', 'li', 'le', 'gli', 'ne', 'il', 'un', 'uno', 'una', 'ma', 'ed', 'se', 'perché', 'anche', 'come', 'dov', 'dove', 'che', 'chi', 'cui', 'non', 'più', 'quale', 'quanto', 'quanti', 'quanta', 'quante', 'quello', 'quelli', 'quella', 'quelle', 'questo', 'questi', 'questa', 'queste', 'si', 'tutto', 'tutti', 'a', 'c', 'e', 'i', 'l', 'o', 'ho', 'hai', 'ha', 'ab

In [26]:
import warnings
warnings.filterwarnings("ignore")


In [27]:
from sklearn.feature_extraction.text import CountVectorizer

* Vediamo una prima esecuzione dell' algoritmo:

volutamente, la fase di processamento del testo, è stata ridotta al minimo indispensabile (ovvero considerando la sola rimozione delle stopwords)

In [28]:
bow=CountVectorizer(max_features=10000,stop_words=italian_stopwords,lowercase=True)
X=bow.fit_transform(corpus)
print(X.shape)

(20058, 10000)


In [29]:
features = bow.get_feature_names_out()
print("vediamo tutte le features: ", features,'\n')
print("vediamone una decina di features: \n", features[1000:1010], '\n')  # stampiamone una decina
print("vediamo la feature n.245: ", features[245], '\n')  # stampiamone una decina

vediamo tutte le features:  ['00' '000' '01' ... 'ztl' 'zurbaran' 'zza'] 

vediamone una decina di features: 
 ['babbo' 'babila' 'bacino' 'bacio' 'badge' 'bagagli' 'bagaglio' 'bagnato'
 'bagni' 'bagno'] 

vediamo la feature n.245:  addosso 



In [30]:
from sklearn.decomposition import LatentDirichletAllocation as LDA

* la cella che segue impiega circa 7 minuti e mezzo per essere eseguita

In [48]:
from sklearn.decomposition import LatentDirichletAllocation as LDA

# fissiamo a-priori il numero di topics (10) ed il numero di iterazioni
n_topics = 10

lda = LDA(n_components=n_topics, max_iter=100,random_state=100, verbose=False)
new_lda = lda.fit_transform(X)

In [49]:
print('Document Topic Matrix: \n', np.array(lda), '\n')
print(new_lda, '\n')
print(len(new_lda), '\n')

print("ecco la distribuzione di probabilità sui topic del primo articolo: \n", np.array(new_lda[0]),'\n')
print("la somma delle probabiltà è: ", np.sum(new_lda[0]),'\n')

Document Topic Matrix: 
 LatentDirichletAllocation(max_iter=100, random_state=100, verbose=False) 

[[0.00208402 0.00208378 0.00208359 ... 0.0020837  0.00208372 0.08586779]
 [0.00833397 0.00833437 0.00833438 ... 0.00833358 0.00833611 0.92499264]
 [0.00285742 0.00285767 0.00285735 ... 0.00285763 0.06480793 0.00285727]
 ...
 [0.18662397 0.00476241 0.00476228 ... 0.00476236 0.00476278 0.77527635]
 [0.01       0.01000101 0.13757968 ... 0.78241598 0.01000097 0.0100002 ]
 [0.00357213 0.00357219 0.00357204 ... 0.00357218 0.00357269 0.00357194]] 

20058 

ecco la distribuzione di probabilità sui topic del primo articolo: 
 [0.00208402 0.00208378 0.00208359 0.0020834  0.00208362 0.00208375
 0.89746265 0.0020837  0.00208372 0.08586779] 

la somma delle probabiltà è:  1.0 



In [50]:
print(new_lda.shape)

(20058, 10)


lda_components è invece la matrice topic x words , in cui per  ogni riga viene espressa la frequenza (ratio) di tutte le parole presenti nel vocabolario prestabilito all'interno del topic indicizzato dal numero di riga. quindi il numero di righe coincide con  numero di topic settato come **iperparametro**

lda.components_[i][j] è la **ratio** della parola j nel topic i, ovvero il numero di volte che la j-esima parola viene attribuita al topic i

In [51]:
print(len(lda.components_))    # numero dei topics: 10
print(len(lda.components_[0])) # numero delle word del vocabolario: 10000
print("Topic Word Matrix: \n")
j=0
for i  in lda.components_:
    print("topic ", j+1, i, '\n')
    j+=1

10
10000
Topic Word Matrix: 

topic  1 [ 0.1000175  11.68171953  0.10000892 ...  0.10002826  0.10003298
  0.10002628] 

topic  2 [0.10000783 0.1000251  4.099978   ... 0.10003348 0.1        0.1       ] 

topic  3 [0.10000953 0.10001462 0.1        ... 0.1        0.1        0.10001935] 

topic  4 [0.10000627 0.10001608 0.1        ... 0.10001268 0.10003079 0.1       ] 

topic  5 [0.10002402 5.51804648 0.1        ... 0.1        0.1        0.10002002] 

topic  6 [0.10001188 0.10003877 0.10000697 ... 1.09994987 4.09981879 0.1       ] 

topic  7 [22.05772755  0.1000005   0.1        ...  2.09997571  0.10000331
  0.10009368] 

topic  8 [0.10000344 0.10010728 0.1        ... 0.1        0.1        8.09982752] 

topic  9 [0.10001537 0.10001842 0.1        ... 0.1        0.1        0.10000782] 

topic  10 [2.14142177e+02 1.00013211e-01 1.00006114e-01 ... 1.00000000e-01
 1.00114132e-01 1.00005318e-01] 



In [52]:
print(lda.components_.shape)

(10, 10000)


* Calcoliamo ora, a titolo di esempio, il "peso"(inteso come misura della frequenza) di una delle feature all'interno del primo topic

In [53]:
feature_100 = bow.get_feature_names_out()[1000]
print(feature_100)

babbo


In [54]:
freq_sul_topic_0=lda.components_[0]
#Questo è il vettore con i pesi di tutte le parole nel ptimo topic
print(freq_sul_topic_0)

[ 0.1000175  11.68171953  0.10000892 ...  0.10002826  0.10003298
  0.10002628]


In [55]:
print(f"il peso della parola {features[1000]} nel primo topic è {freq_sul_topic_0[1000]}")

il peso della parola babbo nel primo topic è 0.10000189808009924


In [56]:
for i in range(len(lda.components_)):
  print(f"il peso di  {features[1000]} nel topic {i+1} è {lda.components_[i][1000]}")

il peso di  babbo nel topic 1 è 0.10000189808009924
il peso di  babbo nel topic 2 è 0.100003455357653
il peso di  babbo nel topic 3 è 0.10000091611346046
il peso di  babbo nel topic 4 è 0.10000000033896392
il peso di  babbo nel topic 5 è 0.10000091380335839
il peso di  babbo nel topic 6 è 7.0999414676620525
il peso di  babbo nel topic 7 è 0.10002946843520345
il peso di  babbo nel topic 8 è 0.10000000016051364
il peso di  babbo nel topic 9 è 0.10000171948944424
il peso di  babbo nel topic 10 è 0.10002016052346165


In [57]:
print(f"il peso totale di {features[1000]} in tutti i topics è {lda.components_.sum(axis=0)[1000]}")#somma per colonne

il peso totale di babbo in tutti i topics è 7.999999999964211


* Per ottenere $\mathbb{P}(\text{parola}\mid \text{topic}(k))$, bisogna normalizzare rispetto somma dei pesi assegnati a tutte le parole all'interno dello stesso topic $k$:

$$\mathbb{P}(w_j \mid z_{w_j}=k)=\frac{\text{ldacomponents}_{[k,j]}}{\sum_i \text{ldacomponents}_{[k,i]}}$$

In [58]:
print(f"Quindi la prob. che {features[1000]} sia campionata dal topic 1 è {lda.components_[0][100]/lda.components_[0].sum()}")

Quindi la prob. che babbo sia campionata dal topic 1 è 2.2578376351684304e-06


In [59]:
topic_word_prob = lda.components_ / lda.components_.sum(axis=1)[:, None]


print('la probabilità di ciascuna delle 10000 parole del dizionario di essere campionate  dal topic 1: ')
topic_proba_0 = topic_word_prob[0]
print(topic_proba_0, '\n') # è la probabilità di ciascuna delle 10000 parole del dizionario
print("La somma per riga è \n")                                # di essere campionate dal topic 1
print(np.sum(topic_proba_0),'\n')

la probabilità di ciascuna delle 10000 parole del dizionario di essere campionate  dal topic 1: 
[2.25812594e-06 2.63741771e-04 2.25793205e-06 ... 2.25836879e-06
 2.25847524e-06 2.25832402e-06] 

La somma per riga è 

1.0000000000000002 



In [60]:
print(topic_word_prob.shape)

(10, 10000)


sommando ogni  colonna di topic_word_prob otteniamo un vettore di dimensione 10000 in cui ogni entrata è un valore (non è una probabilità) che ci dà un indicazione di quanto ogni parola è globalmente distribuita tra tutti i topic

In [61]:
topic_word_proba_sum = topic_word_prob.sum(axis=0)
print(topic_word_proba_sum)
print(len(topic_word_proba_sum))

[4.35552522e-03 3.66535509e-04 5.75765569e-05 ... 1.04884821e-04
 2.26040347e-04 2.15175794e-04]
10000


* Adesso estraiamo le parole più popolari di ogni topic e valutiamo il risultato ottenuto

In [62]:
n_words = 10
for index, topic in enumerate(lda.components_):
    # stampiamo gli indici delle prime 10 parole più popolari, usiamo argsort() per ordinarle in senso crescente,
    # quindi prenderemo gli ultimi 10 indici, denotando l'intervallo come [-10:]
    print("\nTOPIC %d - indici delle %d parole più popolari" % (index+1, n_words))
    print([i for i in topic.argsort()[-n_words:]])

    # utilizziamo il dizionario per trovare le parole in base agli indici
    print("\nTOPIC %d - %d parole più popolari (ordinate crescenti)" % (index+1, n_words))
    print([features[i] for i in topic.argsort()[-n_words:]])

    print('______________________________________________________________________________')


TOPIC 1 - indici delle 10 parole più popolari
[np.int64(3711), np.int64(6398), np.int64(8805), np.int64(9850), np.int64(3342), np.int64(6683), np.int64(2224), np.int64(5493), np.int64(2716), np.int64(8532)]

TOPIC 1 - 10 parole più popolari (ordinate crescenti)
['forse', 'peccato', 'stato', 'visto', 'essere', 'poco', 'cosa', 'molto', 'dire', 'solo']
______________________________________________________________________________

TOPIC 2 - indici delle 10 parole più popolari
[np.int64(5493), np.int64(3473), np.int64(1718), np.int64(1142), np.int64(9708), np.int64(9974), np.int64(5734), np.int64(1550), np.int64(6265), np.int64(6248)]

TOPIC 2 - 10 parole più popolari (ordinate crescenti)
['molto', 'fare', 'città', 'bici', 'verde', 'york', 'new', 'central', 'park', 'parco']
______________________________________________________________________________

TOPIC 3 - indici delle 10 parole più popolari
[np.int64(9811), np.int64(9641), np.int64(1718), np.int64(4608), np.int64(9916), np.int64(98

* Definiamo una funzione per classificare nuove rewiew all'interno di questi topic che abbiamo ricavato

In [63]:


def classify(text, return_proba=False):
    x = bow.transform([text]) # utilizziamo il metodo transform del bow sfruttando il fitting eseguito precedentemente, ricordiamo che l'input deve essere un array
    y_proba = lda.transform(x)[0] # siccome restituisce un array di array, prendiamo solo la prima componente con le propabilità di appartenza a ciascuno dei 25 topics
    y = y_proba.argmax()

    if(return_proba):
        y_proba_p = np.sort(y_proba)
        print("y_proba in senso crescente: \n", y_proba_p, "\n")

        topics_orderedp = np.argsort(y_proba)
        print("topics_ordered in senso crescente: \n", topics_orderedp, "\n")

        topics_ordered = np.argsort(-y_proba)
        print("topics_ordered  in senso decrescente: \n", topics_ordered, "\n")

        y_proba = -np.sort(-y_proba)
        print("y_proba in senso decrescente: \n", y_proba, "\n")

        y_dict = dict(["TOPIC: "+str(topic+1), proba] for topic, proba in zip(topics_ordered, y_proba))
        return y, y_dict

    return y

In [64]:
my_rewiew = "cappella spettacolare, gli  affreschi delle vergini  mi hanno fatto sentire in contatto spirituale con Dio"
y, y_proba = classify(my_rewiew, return_proba=True)

print("Topic di appartenenza: %d" % (y+1))
print("\n")
for topic in y_proba:
    print("%s = %.4f" % (topic, y_proba[topic]))


y_proba in senso crescente: 
 [0.01111199 0.01111232 0.01111242 0.01111248 0.01111407 0.0111142
 0.01111543 0.01111596 0.16739102 0.74370012] 

topics_ordered in senso crescente: 
 [6 7 3 9 8 1 0 2 5 4] 

topics_ordered  in senso decrescente: 
 [4 5 2 0 1 8 9 3 7 6] 

y_proba in senso decrescente: 
 [0.74370012 0.16739102 0.01111596 0.01111543 0.0111142  0.01111407
 0.01111248 0.01111242 0.01111232 0.01111199] 

Topic di appartenenza: 5


TOPIC: 5 = 0.7437
TOPIC: 6 = 0.1674
TOPIC: 3 = 0.0111
TOPIC: 1 = 0.0111
TOPIC: 2 = 0.0111
TOPIC: 9 = 0.0111
TOPIC: 10 = 0.0111
TOPIC: 4 = 0.0111
TOPIC: 8 = 0.0111
TOPIC: 7 = 0.0111


In [65]:
my_second_rewiew="La zona dei giardini completamente immersa nel verde è quella che più mi ha affascinato"
y1, y_proba1 = classify(my_second_rewiew, return_proba=True)

print("Topic di appartenenza: %d" % (y1+1))
print("\n")
for topic in y_proba1:
    print("%s = %.4f" % (topic, y_proba1[topic]))

y_proba in senso crescente: 
 [0.01428571 0.01428571 0.01428601 0.01428717 0.01428866 0.01428983
 0.01429611 0.17106471 0.25724113 0.47167495] 

topics_ordered in senso crescente: 
 [0 2 9 5 7 8 4 6 3 1] 

topics_ordered  in senso decrescente: 
 [1 3 6 4 8 7 5 9 2 0] 

y_proba in senso decrescente: 
 [0.47167495 0.25724113 0.17106471 0.01429611 0.01428983 0.01428866
 0.01428717 0.01428601 0.01428571 0.01428571] 

Topic di appartenenza: 2


TOPIC: 2 = 0.4717
TOPIC: 4 = 0.2572
TOPIC: 7 = 0.1711
TOPIC: 5 = 0.0143
TOPIC: 9 = 0.0143
TOPIC: 8 = 0.0143
TOPIC: 6 = 0.0143
TOPIC: 10 = 0.0143
TOPIC: 3 = 0.0143
TOPIC: 1 = 0.0143


In [66]:
my_third_rewiew="La gita è stata memorabile, i mezzi di trasporto molto efficienti"
y2, y2_proba = classify(my_third_rewiew, return_proba=True)

print("Topic di appartenenza: %d" % (y2+1))
print("\n")
for topic in y2_proba:
    print("%s = %.4f" % (topic, y2_proba[topic]))

y_proba in senso crescente: 
 [0.01250073 0.01250125 0.01250128 0.01250167 0.01250206 0.01250209
 0.01250358 0.01250508 0.16112666 0.73885559] 

topics_ordered in senso crescente: 
 [5 4 3 2 7 1 0 8 9 6] 

topics_ordered  in senso decrescente: 
 [6 9 8 0 1 7 2 3 4 5] 

y_proba in senso decrescente: 
 [0.73885559 0.16112666 0.01250508 0.01250358 0.01250209 0.01250206
 0.01250167 0.01250128 0.01250125 0.01250073] 

Topic di appartenenza: 7


TOPIC: 7 = 0.7389
TOPIC: 10 = 0.1611
TOPIC: 9 = 0.0125
TOPIC: 1 = 0.0125
TOPIC: 2 = 0.0125
TOPIC: 8 = 0.0125
TOPIC: 3 = 0.0125
TOPIC: 4 = 0.0125
TOPIC: 5 = 0.0125
TOPIC: 6 = 0.0125


In [67]:
my_fourth_rewiew="In treno era possibile portare con se la bicicletta, in modo da visitare il paese alto senza faticare troppo"
y3, y3_proba = classify(my_fourth_rewiew, return_proba=True)

print("Topic di appartenenza: %d" % (y3+1))
print("\n")
for topic in y3_proba:
    print("%s = %.4f" % (topic, y3_proba[topic]))

y_proba in senso crescente: 
 [0.00909199 0.00909209 0.00909295 0.00909302 0.0090933  0.00909426
 0.16495362 0.17488875 0.29401535 0.31158466] 

topics_ordered in senso crescente: 
 [5 4 3 9 6 8 7 2 0 1] 

topics_ordered  in senso decrescente: 
 [1 0 2 7 8 6 9 3 4 5] 

y_proba in senso decrescente: 
 [0.31158466 0.29401535 0.17488875 0.16495362 0.00909426 0.0090933
 0.00909302 0.00909295 0.00909209 0.00909199] 

Topic di appartenenza: 2


TOPIC: 2 = 0.3116
TOPIC: 1 = 0.2940
TOPIC: 3 = 0.1749
TOPIC: 8 = 0.1650
TOPIC: 9 = 0.0091
TOPIC: 7 = 0.0091
TOPIC: 10 = 0.0091
TOPIC: 4 = 0.0091
TOPIC: 5 = 0.0091
TOPIC: 6 = 0.0091


In [68]:
my_fifth_rewiew="Stoccolma è magia pura"
y4, y4_proba = classify(my_fifth_rewiew, return_proba=True)

print("Topic di appartenenza: %d" % (y4+1))
print("\n")
for topic in y4_proba:
    print("%s = %.4f" % (topic, y4_proba[topic]))

y_proba in senso crescente: 
 [0.03333333 0.03333333 0.03333333 0.03333333 0.03333415 0.03333532
 0.03333654 0.0333388  0.03335186 0.69997   ] 

topics_ordered in senso crescente: 
 [9 7 2 8 6 0 4 1 3 5] 

topics_ordered  in senso decrescente: 
 [5 3 1 4 0 6 8 2 7 9] 

y_proba in senso decrescente: 
 [0.69997    0.03335186 0.0333388  0.03333654 0.03333532 0.03333415
 0.03333333 0.03333333 0.03333333 0.03333333] 

Topic di appartenenza: 6


TOPIC: 6 = 0.7000
TOPIC: 4 = 0.0334
TOPIC: 2 = 0.0333
TOPIC: 5 = 0.0333
TOPIC: 1 = 0.0333
TOPIC: 7 = 0.0333
TOPIC: 9 = 0.0333
TOPIC: 3 = 0.0333
TOPIC: 8 = 0.0333
TOPIC: 10 = 0.0333


In [69]:
my_sixth_rewiew="parchi, chiese, musei, arte, natura, bici-friendly, esperienze e visite guidate: questa città offre qualsiasi cosa tu voglia; è completa in tutto"
y5, y5_proba = classify(my_sixth_rewiew, return_proba=True)

print("Topic di appartenenza: %d" % (y5+1))
print("\n")
for topic in y5_proba:
    print("%s = %.4f" % (topic, y5_proba[topic]))

y_proba in senso crescente: 
 [0.00625039 0.00625093 0.00625144 0.00625213 0.00625249 0.00625253
 0.00625413 0.11194829 0.29808035 0.54620731] 

topics_ordered in senso crescente: 
 [5 6 3 0 7 2 8 4 9 1] 

topics_ordered  in senso decrescente: 
 [1 9 4 8 2 7 0 3 6 5] 

y_proba in senso decrescente: 
 [0.54620731 0.29808035 0.11194829 0.00625413 0.00625253 0.00625249
 0.00625213 0.00625144 0.00625093 0.00625039] 

Topic di appartenenza: 2


TOPIC: 2 = 0.5462
TOPIC: 10 = 0.2981
TOPIC: 5 = 0.1119
TOPIC: 9 = 0.0063
TOPIC: 3 = 0.0063
TOPIC: 8 = 0.0063
TOPIC: 1 = 0.0063
TOPIC: 4 = 0.0063
TOPIC: 7 = 0.0063
TOPIC: 6 = 0.0063


# Considerazioni

Possiamo vedere come le parole più rappresentative di ciascuno dei 10 topic, solo in alcuni casi diano informazioni utili per ricostruire l'argomento: ad esempio il topic 2 con le parole : "parco","central","park","verde" raggruppa tutte qulle recensioni in cui sono state fornite descrizioni riguardanti la visita dei parchi e delle zone verdi (in particolare Central Park) . D' altro canto, in altri casi come nel topic 1, la maggior parte delle parole più popolari selezionate sono aggettivi o avverbi ("molto","forse","poco","solo"), e perciò non permettono di ricostruire bene l'argomento trattato nelle recensioni.
Quindi le problematiche principali sono state:
 * la scelta di selezionare 10 come numero di topic da scovare (iperparametro) : alcuni topic ricostruiti sembrano essere ridondanti visto che selezionano le stesse parole chiave
 * il processamento ridotto alla sola esclusione delle stop words, ha permesso  di tener conto di tutte quelle parole (in particolare aggettivi e avverbi) utilizzate per enfatizzare emozioni e giudizi, che sono presenti nella maggior parte delle rewiew (ad esempio "molto").

 Per quanto riguarda la classificazione di nuove recensioni, attraverso la funzione appositamente definita, l'algoritmo si comporta abbatanza bene  quando la rewiew fornita in input contiene specifiche parole chiave selezionate unicamente nel topic di riferimento (ad esempio my_second_rewiew con un focus su "giardini" e "zone verdi"), mentre fa più fatica quando la recensione non contiene le parole chiave estratte o ne mischia diverse selezionate in topic differenti.

* Sulla base degli accorgimenti appena detti, modifichiamo il codice introducendo un processamento del testo più selettivo e diminuendo a 8 il numero di topic da ricavare

* Usiamo la libreria spacy per un processamento del testo più accurato. In particolare implementiamo una routine di processeamneto pre-addestrata su un grande dataset contenente migliaia di notizie in lingua italiana

In [70]:
!python -m spacy download it_core_news_sm

     ---------------------------------------- 0.0/13.0 MB ? eta -:--:--
     -------------- ------------------------- 4.7/13.0 MB 25.5 MB/s eta 0:00:01
     ------------------------------------- - 12.6/13.0 MB 31.8 MB/s eta 0:00:01
     --------------------------------------- 13.0/13.0 MB 27.9 MB/s eta 0:00:00
[+] Download and installation successful
You can now load the package via spacy.load('it_core_news_sm')


In [71]:
nlp = spacy.load("it_core_news_sm")
print("Modello caricato correttamente ")

Modello caricato correttamente 


In [72]:
!pip install wordfreq

   ---------------------------------------- 0.0/56.8 MB ? eta -:--:--
   --- ------------------------------------ 5.2/56.8 MB 29.0 MB/s eta 0:00:02
   ----- ---------------------------------- 7.3/56.8 MB 17.6 MB/s eta 0:00:03
   ----- ---------------------------------- 8.4/56.8 MB 15.6 MB/s eta 0:00:04
   ------- -------------------------------- 10.5/56.8 MB 13.4 MB/s eta 0:00:04
   -------- ------------------------------- 12.6/56.8 MB 12.5 MB/s eta 0:00:04
   ---------- ----------------------------- 14.7/56.8 MB 11.8 MB/s eta 0:00:04
   ----------- ---------------------------- 16.3/56.8 MB 11.1 MB/s eta 0:00:04
   ------------ --------------------------- 17.8/56.8 MB 11.3 MB/s eta 0:00:04
   ------------- -------------------------- 19.4/56.8 MB 10.5 MB/s eta 0:00:04
   -------------- ------------------------- 20.4/56.8 MB 9.8 MB/s eta 0:00:04
   ---------------- ----------------------- 23.1/56.8 MB 10.3 MB/s eta 0:00:04
   ---------------- ----------------------- 24.1/56.8 MB 10.0 MB/

In [73]:
from wordfreq import zipf_frequency
#La funzione zipf_frequency serve a stimare la frequenza di una parola in lingua naturale in base a quanto “comune” o “rara” è quella parola.

In [74]:
def clean_text(text):
    doc = nlp(text.lower())
    token_puliti = []

    for tok in doc:
        if tok.is_stop: #rimozione stopwords
            continue
        if tok.pos_ in ["ADV", "ADJ"]:  # rimuove avverbi e aggettivi
            continue
        if tok.is_punct or tok.is_space or tok.like_num: #rimozione spazi punteggiatura e numeri(anche scritti a lettere)
            continue
        if not tok.is_alpha: #manenimento token che contengono solo lettere
            continue
        if not zipf_frequency(tok.lemma_, 'it') > 2: #rimozione parole troppo rare
            continue

        token_puliti.append(tok.lemma_) # lemmatizzazione

    return " ".join(token_puliti)


* Apllichiamo questa funzione di pre-processamneto del testo al nostro corpus (circa 5 minuti per l'esecuzione)

In [75]:
corpus_cleam=[clean_text(document)for document in corpus]


* Oltre ad utilizzare il testo ripulito  tramite la routine pre-addestrata di Spacy, con max_df=1000 rimuoviamo i vocaboli che compaiono in più di 1000 documenti, e con min_df=2 rimuoviamo quelle parole troppo rare all'interno del corpus (quelle che compaiono una sola volta - 1 di document frequency). Inoltre aggiungiamo alle stopwords una lista che inseriamo a mano contenente tutte quelle parole utilizzate per enfatizzare i giudizi.

In [76]:


parole_per_enfatizzare = [
    "molto", "troppo", "poco", "più", "meno", "abbastanza",
    "niente", "nulla", "tanto",  "molti", "poche",
    "stupendo", "bellissimo", "bruttissimo", "magnifico",
    "grazie", "prego", "perfetto", "orribile", "fantastico",
    "meglio", "peggio", "ottimo", "buono", "buona","magico","incredibile","incantevole","emozionante","sicuramente","certamente","peccato","dispiace","piacevole","negativa","postiva","tutto","bello","brutto","assolutamente","stupefacente","sensazionale","emozionante","quando","senza","mai","vale","pena","grande","solo","dopo","poi","mentre","durante","senza","sempre","ogni","vedere","entrare"
]

stopwords_finali = list(set(
    italian_stopwords
    + parole_per_enfatizzare

))


In [77]:
bow1=CountVectorizer(max_features=25000,max_df=1000,min_df=2,stop_words=stopwords_finali,lowercase=True)
X1=bow1.fit_transform(corpus_cleam)
print(X1.shape)

(20058, 6769)


* Dopo questi passaggi sono rimaste 6769 features

In [78]:
features_cl = bow1.get_feature_names_out()
print("vediamo tutte le features: ", features_cl,'\n')
print("vediamone una decina di features: \n", features_cl[1000:1010], '\n')  # stampiamone una decina
print("vediamo la feature n.245: ", features_cl[245], '\n')  # stampiamone una decina

vediamo tutte le features:  ['abbagliare' 'abbandonare' 'abbandonata' ... 'zoo' 'zoom' 'ztl'] 

vediamone una decina di features: 
 ['capitello' 'capitolo' 'capo' 'capodanno' 'capogiro' 'capolavoro'
 'capolinea' 'capolino' 'capoluogo' 'cappa'] 

vediamo la feature n.245:  america 



* Anche per questa cella c'è da attendere alcuni minuti (circa 6)

In [89]:
n_topics = 8

lda1 = LDA(n_components=n_topics, max_iter=150,random_state=31, verbose=False)
#random_state serve per la riproducibilità dei risultati
# con verbose=False disattivo le scritte a schermo relative alle varie esecuzioni
new_lda1 = lda1.fit_transform(X1)

In [90]:
print("Ecco la matrice document topic: \n")
print(new_lda1, '\n')

Ecco la matrice document topic: 

[[0.00694866 0.71787843 0.00694612 ... 0.00694768 0.00694623 0.00696014]
 [0.01785998 0.01785798 0.87497435 ... 0.01787205 0.01785714 0.01785714]
 [0.00658794 0.79995822 0.00657907 ... 0.00658139 0.10032115 0.00658084]
 ...
 [0.00962026 0.00961743 0.93266309 ... 0.00961746 0.00961577 0.00962499]
 [0.01562767 0.01563009 0.01563015 ... 0.17380213 0.7323947  0.015641  ]
 [0.00962374 0.93265115 0.00961957 ... 0.00961586 0.00962991 0.00961582]] 



In [91]:
print("Ecco ora la matrice topic word: \n")
j=0
for i in range(len(lda1.components_)):
   print(f"topic {j+1}: {lda1.components_[i]}")
   print("\n")
   j=j+1
print(f"Questa matrice matrice ha dimensioni {lda1.components_.shape}")

Ecco ora la matrice topic word: 

topic 1: [1.25001930e-01 5.88929863e+00 1.25000018e-01 ... 3.68124838e+02
 2.12491148e+00 3.12417984e+00]


topic 2: [0.12500001 0.12538475 0.12782941 ... 0.12501834 0.12500001 0.12513043]


topic 3: [0.12501088 0.12505339 1.12323504 ... 0.12502675 0.12500002 0.12500002]


topic 4: [0.12503    0.12502832 0.12500005 ... 0.12501667 0.12500003 0.12505023]


topic 5: [11.12479216  3.32425005  0.12500003 ...  0.12500616  0.12500002
  0.12500001]


topic 6: [0.12502098 0.12527496 0.12515531 ... 0.12501139 0.12508838 0.12500002]


topic 7: [ 0.12504487 16.16058807  0.12500008 ...  0.1250377   0.12500004
  0.12552467]


topic 8: [0.12509916 0.12512183 1.12378007 ... 0.12504501 0.12500002 0.12511479]


Questa matrice matrice ha dimensioni (8, 6769)


In [92]:
print(f"La probabilità che la parola {features_cl[3456]} appartenga al topic {len(lda1.components_)} è {lda1.components_[7][3456]/lda1.components_[7].sum()}")

La probabilità che la parola mago appartenga al topic 8 è 5.4943936015419955e-06


In [93]:
max=8
for index, topic in enumerate(lda1.components_):
    # stampiamo gli indici delle prime 10 parole più popolari, usiamo argsort() per ordinarle in senso crescente,
    # quindi prenderemo gli ultimi 10 indici, denotando l'intervallo come [-10:]
    print("\nTOPIC %d - indici delle %d parole più popolari" % (index+1, max))
    print([i for i in topic.argsort()[-max:]])

    # utilizziamo il dizionario per trovare le parole in base agli indici
    print("\nTOPIC %d - %d parole più popolari (ordinate crescenti)" % (index+1, max))
    print([features_cl[i] for i in topic.argsort()[-max:]])

    print('______________________________________________________________________________')


TOPIC 1 - indici delle 8 parole più popolari
[np.int64(2775), np.int64(5422), np.int64(3239), np.int64(2421), np.int64(4446), np.int64(772), np.int64(2703), np.int64(2700)]

TOPIC 1 - 8 parole più popolari (ordinate crescenti)
['grattacielo', 'scoiattolo', 'laghetto', 'film', 'polmone', 'bicicletta', 'giro', 'girare']
______________________________________________________________________________

TOPIC 2 - indici delle 8 parole più popolari
[np.int64(964), np.int64(2631), np.int64(3831), np.int64(3832), np.int64(670), np.int64(5171), np.int64(5547), np.int64(3347)]

TOPIC 2 - 8 parole più popolari (ordinate crescenti)
['canale', 'gente', 'navigli', 'naviglio', 'bar', 'ristorante', 'sera', 'locale']
______________________________________________________________________________

TOPIC 3 - indici delle 8 parole più popolari
[np.int64(4745), np.int64(2812), np.int64(4507), np.int64(3437), np.int64(1271), np.int64(2417), np.int64(779), np.int64(3062)]

TOPIC 3 - 8 parole più popolari (ordi

* Il risultato è decisamente più soddisfacente : ad eccezione del topic 8 che "rimane un pò vago" , le parole chiave ricavate, sembrano inquadrare un argomento specifico tra quelli trattati nelle varie recensioni.
- 3 Arte e Musei
- 2 Movida milanese nella zona dei Navigli
- 1 Attività nel verde urbano
- 5 Turismo religioso
- 4 Location panoramiche
- 7 Itinerari turistici
- 6 Visite guidate


* Adesso ripetiamo gli stessi passaggi per i titoli delle recensioni, analizzando il caso delle recensioni postive (da 3 stelle in su) e di quelle negative (massimo 2 stelle)

* Obiettivo:  Valutare se l'algorimo di topic extraction  a partire dal modello LDA è efficace per scopi di sentiment classification

In [94]:
title_str_positive=df[df["stars"]>=3]["title"].astype(str)
title_str_positive

0                                    Zona del canal grande
1                                        Visita di piacere
2                                       Da vedere a Milano
3                             Il Duomo sempre affascinante
4                                              Eccezionale
                               ...                        
20053                                        È bellissimo!
20054                                             Navigli 
20055                    Uno dei musei più belli del mondo
20056    Spettacolare Porta simbolo della città di Berl...
20057    Fantastico angolo di Milano che assolutamente ...
Name: title, Length: 19710, dtype: object

In [95]:
title_str_negative=df[df["stars"]<3]["title"].astype(str)
title_str_negative

10          Giochi d'acqua natalizi e navigazione musicale
15                                       Non è il migliore
23                                                  Noioso
88                                          Caratteristico
113                                          Niente di che
                               ...                        
19506    Tristissima imitazione di un ambiente del passato
19656              bello e caratteristico....ma non sempre
19727                                            Delusione
19856                         Solo per anglofoni, peccato!
19861                                              Mah....
Name: title, Length: 348, dtype: object

*  in questo caso manteniamo aggettivi e avverbi che, come abbiamo detto, sono fondamentali per conferire enfasi ai giudizi

In [96]:
def clean_titles(text):
    doc = nlp(text.lower())
    token_puliti = []

    for tok in doc:
        if tok.is_stop: #rimozione stopwords
            continue
       # if tok.pos_ in ["ADV", "ADJ"]:  # in questo caso manteniamo aggettivi e avverbi che, come abbiamo detto, sono fondamentali per conferire enfasi ai giudizi
         #   continue
        if tok.is_punct or tok.is_space or tok.like_num: #rimozione spazi punteggiatura e numeri(anche scritti a lettere)
            continue
        if not tok.is_alpha: #manenimento token che contengono solo lettere
            continue
        if not zipf_frequency(tok.lemma_, 'it') > 2: #rimozione parole troppo rare
            continue

        token_puliti.append(tok.lemma_) # lemmatizzazione

    return " ".join(token_puliti)


In [97]:
title_pos_clean=[clean_titles(t) for t in title_str_positive]
title_neg_clean=[clean_titles(t) for t in title_str_negative]

In [113]:
bowtitlepos=CountVectorizer(max_features=5000,stop_words=italian_stopwords,max_df=0.8,min_df=2 ,lowercase=True)
Xtitlepos=bowtitlepos.fit_transform(title_pos_clean)
print(Xtitlepos.shape)

(19710, 1535)


In [114]:
title_feature_pos=bowtitlepos.get_feature_names_out()
print(f"Ecco tutte le feature: {title_feature_pos}")
print(f"Ecco una decina di feature: {title_feature_pos[1000:1010]}")
print(f"Ecco la feature numero 245: {title_feature_pos[245]}")

Ecco tutte le feature: ['abbraccio' 'abitare' 'abito' ... 'zona' 'zonzo' 'zoo']
Ecco una decina di feature: ['percorrere' 'percorso' 'perdere' 'pere' 'perfetto' 'perfezione'
 'periodo' 'perla' 'personale' 'personalmente']
Ecco la feature numero 245: colorare


In [115]:
number_of_topics=5 #diminuiamo a 5 l'iperparametro
lda_title_pos=LDA(n_components=number_of_topics,max_iter=200,random_state=56,verbose=False)
new_lda_title_pos=lda_title_pos.fit_transform(Xtitlepos)

In [116]:
print("Ecco la matrice document topic: \n")
print(new_lda_title_pos, '\n')

Ecco la matrice document topic: 

[[0.0679074  0.06666754 0.39883557 0.06666762 0.39992187]
 [0.06666695 0.066667   0.4000427  0.06666702 0.39995633]
 [0.06666669 0.06666669 0.06666669 0.40000127 0.39999865]
 ...
 [0.73055623 0.06666673 0.06944357 0.06666673 0.06666674]
 [0.37413272 0.04000032 0.23987865 0.30598791 0.0400004 ]
 [0.19999592 0.03333344 0.20000685 0.36669707 0.19996671]] 



In [117]:
print("Ecco ora la matrice topic word: \n")
j=0
for i in range (len(lda_title_pos.components_)):
   print(f"topic {j+1}: {lda_title_pos.components_[i]}")
   print("\n")
   j=j+1

Ecco ora la matrice topic word: 

topic 1: [ 0.20003211  0.20001933  0.20001395 ... 36.38089256  0.20001731
  0.20001259]


topic 2: [0.20003634 0.20292302 0.20001578 ... 0.20091966 0.20001934 0.20001419]


topic 3: [ 0.20003785  0.20002322  0.20001644 ... 62.01483485  0.20002013
  0.20095858]


topic 4: [2.19984964 2.19633956 2.1999347  ... 0.20254722 0.20002103 0.20001577]


topic 5: [0.20004406 0.20069487 0.20001913 ... 0.20080571 3.1999222  7.19899887]




In [118]:
print(f"La probabilità che la parola {title_feature_pos[567]} appartenga al topic {len(lda_title_pos.components_)} è {lda_title_pos.components_[4][567]/ lda_title_pos.components_[4].sum()} ")

La probabilità che la parola grattacielo appartenga al topic 5 è 3.33124402771373e-05 


In [119]:
max=8
for index, topic in enumerate(lda_title_pos.components_):
    # stampiamo gli indici delle prime 10 parole più popolari, usiamo argsort() per ordinarle in senso crescente,
    # quindi prenderemo gli ultimi 10 indici, denotando l'intervallo come [-10:]
    print("\nTOPIC %d - indici delle %d parole più popolari" % (index+1, max))
    print([i for i in topic.argsort()[-max:]])

    # utilizziamo il dizionario per trovare le parole in base agli indici
    print("\nTOPIC %d - %d parole più popolari (ordinate crescenti)" % (index+1, max))
    print([title_feature_pos[i] for i in topic.argsort()[-max:]])

    print('______________________________________________________________________________')


TOPIC 1 - indici delle 8 parole più popolari
[np.int64(138), np.int64(224), np.int64(979), np.int64(1326), np.int64(536), np.int64(461), np.int64(1353), np.int64(131)]

TOPIC 1 - 8 parole più popolari (ordinate crescenti)
['berlino', 'chiesa', 'passeggiata', 'spettacolare', 'gioiello', 'fantastico', 'storia', 'bello']
______________________________________________________________________________

TOPIC 2 - indici delle 8 parole più popolari
[np.int64(315), np.int64(902), np.int64(85), np.int64(1528), np.int64(876), np.int64(1038), np.int64(791), np.int64(1484)]

TOPIC 2 - 8 parole più popolari (ordinate crescenti)
['cuore', 'oasi', 'arte', 'york', 'new', 'polmone', 'meraviglioso', 'verde']
______________________________________________________________________________

TOPIC 3 - indici delle 8 parole più popolari
[np.int64(811), np.int64(846), np.int64(665), np.int64(603), np.int64(1503), np.int64(978), np.int64(1002), np.int64(1501)]

TOPIC 3 - 8 parole più popolari (ordinate crescent

* Passiamo alle recensioni negative

In [124]:
bowtitleneg=CountVectorizer(max_features=5000,stop_words=italian_stopwords,min_df=2,lowercase=True)
Xtitleneg=bowtitleneg.fit_transform(title_neg_clean)
print(Xtitleneg.shape)

(348, 84)


In [127]:
title_feature_neg=bowtitleneg.get_feature_names_out()
print(f"Ecco tutte le feature: {title_feature_neg}")
print(f"Ecco una decina di feature: {title_feature_neg[70:80]}")
print(f"Ecco la feature numero 74: {title_feature_neg[74]}")

Ecco tutte le feature: ['abbandono' 'accogliente' 'acqua' 'arte' 'aspettare' 'bah' 'bello'
 'berlino' 'biglietto' 'brutto' 'canale' 'caos' 'caratteristico'
 'chiudere' 'chiuso' 'confusione' 'cuore' 'cupola' 'degrado' 'deludente'
 'deludere' 'delusione' 'deluso' 'dispersivo' 'duomo' 'eccezionale'
 'entrare' 'esperienza' 'evitare' 'fare' 'gente' 'interessante' 'locale'
 'madrid' 'mah' 'memoriale' 'meraviglioso' 'migliore' 'milano' 'mortale'
 'movida' 'muro' 'museo' 'natalizio' 'navigli' 'noia' 'noioso' 'occasione'
 'opera' 'organizzazione' 'pagare' 'particolare' 'pensare' 'percorso'
 'personale' 'pessimo' 'piacere' 'poco' 'potere' 'prado' 'prenotazione'
 'ridicolo' 'rumore' 'serata' 'simbolo' 'sopravvalutare' 'sopravvalutato'
 'splendido' 'sporco' 'squallido' 'stare' 'storia' 'tanto' 'totale'
 'trascurare' 'turista' 'unico' 'valorizzare' 'vedere' 'vergogna'
 'viaggio' 'visita' 'visitare' 'zona']
Ecco una decina di feature: ['stare' 'storia' 'tanto' 'totale' 'trascurare' 'turista' 'unico'

In [128]:
number_of_topics=3   #ho decrementato l'iperparametro perchè ci sono molte meno parole nel vocabolario
lda_title_neg=LDA(n_components=number_of_topics,max_iter=200,random_state=30,verbose=False)
new_lda_title_neg=lda_title_neg.fit_transform(Xtitleneg)

In [129]:
print("Ecco la matrice document topic: \n")
print(new_lda_title_neg, '\n')

Ecco la matrice document topic: 

[[0.12044436 0.76778815 0.11176749]
 [0.16693183 0.66608095 0.16698722]
 [0.16722451 0.66543222 0.16734327]
 ...
 [0.16673992 0.66650495 0.16675513]
 [0.33333333 0.33333333 0.33333333]
 [0.16687761 0.66620059 0.1669218 ]] 



In [130]:
print("Ecco ora la matrice topic word: \n")
j=0
for i in range (len(lda_title_neg.components_)):
   print(f"topic {j+1}: {lda_title_neg.components_[i]}")
   print("\n")
   j=j+1

Ecco ora la matrice topic word: 

topic 1: [ 0.33555638  0.33555641  0.33512103  1.30466297  0.33524329  0.33495685
 16.3159862   0.33489324  0.33488616  0.3348922   4.32979559  0.34607519
  3.3298391   0.3354934   0.36119019  2.33123187  2.32922742  0.36437072
  0.35940092  0.3350409   9.32810222  0.3350757   0.33528927  2.32958064
  9.31635933  0.33560611  2.29008807  0.33473746  3.32787369  2.33087636
  2.32959869  2.32922737  0.33517557  0.33419908  0.33544165  0.33534509
  2.32966347  0.33509303  7.44448279  0.33436863  0.34912079  0.3351748
  3.27509324  1.32297883  4.3296697   0.33436863  0.33555637  0.33495859
  1.25912002  0.34025577  0.35060751  0.33549339  0.33534508  0.33456218
  0.33555638  0.33465134  3.32787367  4.31365969  3.33030957  0.33441088
  0.33555643  0.33555638  2.28940406  2.32770625  0.33510036  4.3279588
  4.32931164  0.33510201  0.33500197  0.33534508  2.32770631  0.33500234
  1.32898871  0.33432399  0.33493876  0.34944491  2.3296259   0.3355564
  3.2035640

In [131]:
print(f"La probabilità che la parola {title_feature_neg[74]} appartenga al topic {len(lda_title_neg.components_)} è {lda_title_neg.components_[2][74]/ lda_title_neg.components_[2].sum()} ")

La probabilità che la parola trascurare appartenga al topic 3 è 0.028417734333520638 


In [132]:
max=8
for index, topic in enumerate(lda_title_neg.components_):
    # stampiamo gli indici delle prime 10 parole più popolari, usiamo argsort() per ordinarle in senso crescente,
    # quindi prenderemo gli ultimi 10 indici, denotando l'intervallo come [-10:]
    print("\nTOPIC %d - indici delle %d parole più popolari" % (index+1, max))
    print([i for i in topic.argsort()[-max:]])

    # utilizziamo il dizionario per trovare le parole in base agli indici
    print("\nTOPIC %d - %d parole più popolari (ordinate crescenti)" % (index+1, max))
    print([title_feature_neg[i] for i in topic.argsort()[-max:]])

    print('______________________________________________________________________________')


TOPIC 1 - indici delle 8 parole più popolari
[np.int64(66), np.int64(44), np.int64(10), np.int64(83), np.int64(38), np.int64(24), np.int64(20), np.int64(6)]

TOPIC 1 - 8 parole più popolari (ordinate crescenti)
['sopravvalutato', 'navigli', 'canale', 'zona', 'milano', 'duomo', 'deludere', 'bello']
______________________________________________________________________________

TOPIC 2 - indici delle 8 parole più popolari
[np.int64(7), np.int64(11), np.int64(32), np.int64(41), np.int64(37), np.int64(73), np.int64(34), np.int64(21)]

TOPIC 2 - 8 parole più popolari (ordinate crescenti)
['berlino', 'caos', 'locale', 'muro', 'migliore', 'totale', 'mah', 'delusione']
______________________________________________________________________________

TOPIC 3 - indici delle 8 parole più popolari
[np.int64(68), np.int64(78), np.int64(4), np.int64(9), np.int64(49), np.int64(27), np.int64(81), np.int64(55)]

TOPIC 3 - 8 parole più popolari (ordinate crescenti)
['sporco', 'vedere', 'aspettare', 'brut

* Il modello non è stato molto efficace per scopi di sentiment classification :

Seppure alcune parole che esprimono polarità sono state individuate come parole chiave dei vari topic, queste non permettono di distinguere bene quali sono stati gli aspetti positivi e quelli negativi riscontrati dagli utenti.
Le ragioni si possono attribuire alle seguenti:
 * pochi titoli disponibili per quanto riguarda le recensioni negative
 * Titoli molto brevi

# LDA con codifica Tf-Idf

* Ricreiamo il modello usando questa volta la codifica tf-Idf

In [133]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [134]:
tfidf=TfidfVectorizer(max_features=15000,stop_words=stopwords_finali,min_df=2,max_df=1000)
X2=tfidf.fit_transform(corpus_cleam)
X2.shape

(20058, 6769)

In [135]:
tfidf_features = tfidf.get_feature_names_out()
print("vediamo tutte le features: ", tfidf_features,'\n')
print("vediamo una decina di features: \n", tfidf_features[1000:1010])
print("vediamo la feature n.1245: ", tfidf_features[245])

vediamo tutte le features:  ['abbagliare' 'abbandonare' 'abbandonata' ... 'zoo' 'zoom' 'ztl'] 

vediamo una decina di features: 
 ['capitello' 'capitolo' 'capo' 'capodanno' 'capogiro' 'capolavoro'
 'capolinea' 'capolino' 'capoluogo' 'cappa']
vediamo la feature n.1245:  america


* Soliti 5-6 minuti per l'esecuzione della cella

In [136]:
n_topics=8
lda_tfidf=LDA(n_components=n_topics,max_iter=150,verbose=False,random_state=31)
new_lda_tfidf=lda_tfidf.fit_transform(X2)

In [137]:
print(f"Ecco la matrice document topic: \n")
print(new_lda_tfidf, '\n')

Ecco la matrice document topic: 

[[0.0253017  0.57520745 0.27268309 ... 0.02529873 0.02527344 0.025279  ]
 [0.03709326 0.03707722 0.5553013  ... 0.03713885 0.03707466 0.03708863]
 [0.02424864 0.7868354  0.02421781 ... 0.02422284 0.0242501  0.02422444]
 ...
 [0.02839386 0.02838576 0.801275   ... 0.02838882 0.02838084 0.02838851]
 [0.03550765 0.03551646 0.03551144 ... 0.03551205 0.44061803 0.34625525]
 [0.0289213  0.79759676 0.02891781 ... 0.02891054 0.02890326 0.02890382]] 



In [138]:
print(f"Ecco ora la matrice topic word: \n")
j=0
for i in range (len(lda_tfidf.components_)):
   print(f"topic {j+1}: {lda_tfidf.components_[i]}")
   j+=1
   print("\n")

print(f"Questa matrice ha dimensioni {lda_tfidf.components_.shape}")

Ecco ora la matrice topic word: 

topic 1: [ 0.12501111  2.18812206  0.12500002 ... 89.370671    1.17273365
  0.35921132]


topic 2: [0.12500006 0.12556689 0.70861057 ... 0.12503568 0.12500006 0.12669444]


topic 3: [0.12517907 0.12508107 0.12500162 ... 0.12504292 0.12500008 0.68544573]


topic 4: [0.12508319 0.12512346 0.12500004 ... 0.12500982 0.12500009 0.12500042]


topic 5: [4.04704869 1.66254008 0.12500003 ... 0.12501266 0.12500007 0.12500029]


topic 6: [0.1250001  0.12529102 0.12500005 ... 0.12501022 0.12500011 0.12605868]


topic 7: [0.12500527 4.46901949 0.12503539 ... 0.1250729  0.12500023 0.12500106]


topic 8: [0.12500748 0.12512275 0.12500004 ... 0.12502728 0.12506941 0.39230323]


Questa matrice ha dimensioni (8, 6769)


In [126]:
max=8
for index, topic in enumerate(lda_tfidf.components_):
    # stampiamo gli indici delle prime 10 parole più popolari, usiamo argsort() per ordinarle in senso crescente,
    # quindi prenderemo gli ultimi 10 indici, denotando l'intervallo come [-10:]
    print("\nTOPIC %d - indici delle %d parole più popolari" % (index+1, max))
    print([i for i in topic.argsort()[-max:]])

    # utilizziamo il dizionario per trovare le parole in base agli indici
    print("\nTOPIC %d - %d parole più popolari (ordinate crescenti)" % (index+1, max))
    print([tfidf_features[i] for i in topic.argsort()[-max:]])

    print('______________________________________________________________________________')


TOPIC 1 - indici delle 8 parole più popolari
[np.int64(4602), np.int64(2248), np.int64(3855), np.int64(5934), np.int64(3165), np.int64(585), np.int64(2421), np.int64(1311)]

TOPIC 1 - 8 parole più popolari (ordinate crescenti)
['primavera', 'estate', 'neve', 'stagione', 'inverno', 'autunno', 'film', 'colore']
______________________________________________________________________________

TOPIC 2 - indici delle 8 parole più popolari
[np.int64(2631), np.int64(3832), np.int64(3831), np.int64(346), np.int64(670), np.int64(5547), np.int64(5171), np.int64(3347)]

TOPIC 2 - 8 parole più popolari (ordinate crescenti)
['gente', 'naviglio', 'navigli', 'aperitivo', 'bar', 'sera', 'ristorante', 'locale']
______________________________________________________________________________

TOPIC 3 - indici delle 8 parole più popolari
[np.int64(3240), np.int64(5422), np.int64(3239), np.int64(2775), np.int64(2703), np.int64(4446), np.int64(772), np.int64(2700)]

TOPIC 3 - 8 parole più popolari (ordinate c

* I risultati ottenuti con i due diversi tipi di codifica del corpus di documenti sono differenti (a parità di operazioni di processamento preliminari effettuate sui testi)
* topic differenti da quelli scoperti in precedenza :  
   * 1 Stagioni e Clima
   * 8 Shoah

# Topic Modelling con Gensim

* Riproponiamo di seguito la routine completa spiegata nei notebook condivisi del corso

In [139]:
from gensim.corpora import Dictionary
from gensim.models.ldamodel import LdaModel
from gensim.utils import simple_preprocess

In [129]:
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [141]:
lemmatizer=nltk.stem.WordNetLemmatizer()

In [241]:
def preprocess (text):
  tokens=[]
  for token in simple_preprocess(text):
   if(token not in stopwords_finali):   #come insieme per le stopwords manteniamo quello arricchito con l'aggiunta delle parole che 'intensificano'
     tokens.append(lemmatizer.lemmatize(token, pos='v'))
  return tokens

In [242]:
df_positive=df[df["stars"]>=3]

* Per provare a migliorare il risultato sui titoli, combiniamo la pipeline di processamento di spacy con il 'simple_preprocess' di gensim

In [243]:
def clean1_titles(text):
    doc = nlp(text.lower())
    token_puliti = []

    for tok in doc:
        if tok.is_stop: #rimozione stopwords
            continue
        if tok.pos_ in ["ADV"]:  #proviamo a mantenere solo aggettivi ma non avverbi
            continue
        if tok.is_punct or tok.is_space or tok.like_num: #rimozione spazi punteggiatura e numeri(anche scritti a lettere)
            continue
        if not tok.is_alpha: #manenimento token che contengono solo lettere
            continue
        if not zipf_frequency(tok.lemma_, 'it') > 2: #rimozione parole troppo rare
            continue

        token_puliti.append(tok.lemma_) # lemmatizzazione

    return " ".join(token_puliti)

In [244]:
df_positive["title"]=[clean1_titles(title) for title in df_positive["title"]]

In [245]:
df_positive["title_preprocessed"]=df_positive["title"].map(preprocess)
df_positive.head(20)

,Unnamed: 0,comment,title,date,name,stars,comment_preprocessed,title_preprocessed
0,0,interessante sera trambusto età sacco bar rist...,zona canal,26 ottobre 2016,gfdmmad,5.0,"[interessante, sera, trambusto, età, sacco, ba...","[zona, canal]"
1,1,organizzare coda entrata piantine visita perco...,visita piacere,26 aprile 2019,Diabolik-Eva-014,5.0,"[organizzare, coda, entrata, piantine, visita,...","[visita, piacere]"
2,2,amico o expo strada il navigli porto darsena r...,vedere Milano,18 novembre 2018,O5262KGchrisd,5.0,"[amico, expo, strada, navigli, porto, darsena,...",[milano]
3,3,visitare duomo Milano madonnina guglia esperie...,duomo affascinante,7 agosto 2019,Babellina,5.0,"[visitare, duomo, milano, madonnina, guglia, e...","[duomo, affascinante]"
4,4,vista duomo uscita metro togliere fiato il lav...,eccezionale,9 dicembre 2019,Nik804,5.0,"[vista, duomo, uscita, metro, togliere, fiato,...",[eccezionale]
5,5,Milano visitare chiesa cuore Milano passo univ...,visita imperdibile,21 aprile 2017,matteodifelice,5.0,"[milano, visitare, chiesa, cuore, milano, pass...","[visita, imperdibile]"
6,6,tenere passeggiare stare,oasi caos,23 dicembre 2017,Ghert2015,4.0,"[tenere, passeggiare, stare]","[oasi, caos]"
7,7,creare traccia passato impedire adeguare si es...,sospendere passato presente,10 febbraio 2017,bobmonica,5.0,"[creare, traccia, passato, impedire, adeguare,...","[sospendere, passato, presente]"
8,8,parola descrivere bellezza parco passeggiare c...,imperdibile,6 novembre 2016,Duecuorieunaciccions,5.0,"[parola, descrivere, bellezza, parco, passeggi...",[imperdibile]
9,9,polmone simbolo New York consigliare passeggia...,passeggiata obbligatorio,23 novembre 2017,CDam7,5.0,"[polmone, simbolo, new, york, consigliare, pas...","[passeggiata, obbligatorio]"


In [246]:
titles_p=df_positive["title_preprocessed"].values
print(titles_p.shape)

print(titles_p)
print("\n")
print(titles_p[:3])

(19710,)
[list(['zona', 'canal']) list(['visita', 'piacere']) list(['milano']) ...
 list(['museo']) list(['spettacolare', 'porta', 'simbolo', 'berlino'])
 list(['angolo', 'milano', 'perdere'])]


[list(['zona', 'canal']) list(['visita', 'piacere']) list(['milano'])]


In [247]:
dictionary=Dictionary(titles_p)
for token,id in dictionary.token2id.items():
  print(f"{token} : {id}")

canal : 0
zona : 1
piacere : 2
visita : 3
milano : 4
affascinante : 5
duomo : 6
eccezionale : 7
imperdibile : 8
caos : 9
oasi : 10
passato : 11
presente : 12
sospendere : 13
obbligatorio : 14
passeggiata : 15
domenica : 16
parco : 17
berlino : 18
ragione : 19
smarrimento : 20
relax : 21
sport : 22
minuto : 23
delizia : 24
occhio : 25
altro : 26
wow : 27
unico : 28
meraviglia : 29
meraviglioso : 30
stagione : 31
perdere : 32
gita : 33
gioiello : 34
visitare : 35
gestire : 36
guidare : 37
interessante : 38
surreale : 39
more : 40
wall : 41
età : 42
sbagliare : 43
cuore : 44
new : 45
verde : 46
york : 47
adorare : 48
city : 49
imponente : 50
museo : 51
ny : 52
simbolo : 53
cambiamento : 54
continuo : 55
inaspettato : 56
eccessivo : 57
grattacielo : 58
immenso : 59
caratteristico : 60
sorprendere : 61
natale : 62
pensavamo : 63
sorpresa : 64
pace : 65
superbo : 66
monumento : 67
corsa : 68
orario : 69
cappello : 70
sistina : 71
vero : 72
centro : 73
milanese : 74
movida : 75
faticoso : 76


In [248]:
dictionary.filter_extremes(no_below=3,no_above=0.6,keep_n=2500)


In [249]:
X3=[dictionary.doc2bow(title) for title in titles_p]
print(f"Ecco i primi 3 titoli in formato bow: \n {X3[:3]}")

Ecco i primi 3 titoli in formato bow: 
 [[(0, 1), (1, 1)], [(2, 1), (3, 1)], [(4, 1)]]


In [250]:
from gensim.models import LdaMulticore
lda_gensim_p=LdaMulticore(X3,num_topics=5,id2word=dictionary,passes=250,workers=3)
#4 minuti

In [251]:
for index,topic in lda_gensim_p.print_topics():
  print("\nTopic %d - parole chiave" % (index+1))
  print(topic)


Topic 1 - parole chiave
0.083*"meraviglioso" + 0.052*"museo" + 0.039*"passeggiata" + 0.038*"unico" + 0.033*"relax" + 0.031*"suggestivo" + 0.028*"milanese" + 0.023*"esperienza" + 0.021*"chiesa" + 0.021*"visita"

Topic 2 - parole chiave
0.071*"park" + 0.060*"central" + 0.047*"spettacolare" + 0.045*"berlino" + 0.040*"interessante" + 0.032*"spettacolo" + 0.032*"splendido" + 0.028*"film" + 0.022*"enorme" + 0.020*"giro"

Topic 3 - parole chiave
0.093*"verde" + 0.065*"polmone" + 0.051*"storia" + 0.051*"meraviglia" + 0.046*"imperdibile" + 0.044*"visita" + 0.029*"oasi" + 0.022*"muro" + 0.018*"centro" + 0.017*"manhattan"

Topic 4 - parole chiave
0.129*"milano" + 0.072*"perdere" + 0.055*"visitare" + 0.052*"passeggiare" + 0.048*"new" + 0.047*"york" + 0.038*"cuore" + 0.025*"simbolo" + 0.024*"arte" + 0.019*"capolavoro"

Topic 5 - parole chiave
0.088*"milano" + 0.087*"parco" + 0.076*"duomo" + 0.056*"sistina" + 0.047*"gioiello" + 0.041*"cappello" + 0.039*"immenso" + 0.033*"tappa" + 0.024*"maestoso" +

In [252]:
import pyLDAvis.gensim
lda_gensim_p_vis=pyLDAvis.gensim.prepare(lda_gensim_p,X3 ,dictionary, mds='tsne')
pyLDAvis.display(lda_gensim_p_vis)

* Passiamo alle recensioni negative

In [253]:
df_negative=df[df["stars"]<3]


In [254]:
df_negative["title"]=[clean1_titles(title) for title in df_negative["title"]]

In [255]:
df_negative["title_preprocessed"]=df_negative["title"].map(preprocess)
df_negative

,Unnamed: 0,comment,title,date,name,stars,comment_preprocessed,title_preprocessed
10,10,visitare milano periodo Natale mille rendere p...,gioco acqua natalizio navigazione musicale,15 dicembre 2019,Alessandra d,1.0,"[visitare, milano, periodo, natale, mille, ren...","[gioco, acqua, natalizio, navigazione, musicale]"
15,15,canale esaurire acqua alcuno parte costruzione...,migliore,7 marzo 2013,Tiffany H,2.0,"[canale, esaurire, acqua, alcuno, parte, costr...",[migliore]
23,23,museo maniera visita,noioso,8 agosto 2020,Irene antonacci,2.0,"[museo, maniera, visita]",[noioso]
88,88,zona aggiornare degrado diverso locale negozio...,caratteristico,6 novembre 2017,fyury,2.0,"[zona, aggiornare, degrado, diverso, locale, n...",[caratteristico]
113,113,troppo gente volere vedere piazza piazza capol...,,18 marzo 2019,Francesco,1.0,"[troppo, gente, volere, vedere, piazza, piazza...",[]
...,...,...,...,...,...,...,...,...
19506,19506,milanese generazione vedere sfruttamento quart...,tristissimo imitazione ambiente passato,18 maggio 2016,enmal m,2.0,"[milanese, generazione, vedere, sfruttamento, ...","[tristissimo, imitazione, ambiente, passato]"
19656,19656,locale carino moda perché potenziale,bello caratteristico,10 novembre 2011,Osvaldo D,2.0,"[locale, carino, moda, potenziale]",[caratteristico]
19727,19727,idea portare il turista sistemare argine passa...,delusione,19 agosto 2019,Elisa B,2.0,"[idea, portare, turista, sistemare, argine, pa...",[delusione]
19856,19856,delusione Berlino conoscere o voglia leggere i...,,10 gennaio 2017,coleus,2.0,"[delusione, berlino, conoscere, voglia, legger...",[]


In [256]:
titles_n=df_negative["title_preprocessed"].values
print(titles_n)
print(titles_n.shape)

[list(['gioco', 'acqua', 'natalizio', 'navigazione', 'musicale'])
 list(['migliore']) list(['noioso']) list(['caratteristico']) list([])
 list(['pensare']) list(['sporco']) list([]) list([]) list(['caos'])
 list(['organizzazione', 'inqualificabile']) list([])
 list(['simbolo', 'berlino']) list(['visita', 'guidare'])
 list(['godere', 'zona']) list(['turistico', 'interessante'])
 list(['museo', 'prado']) list(['vergogna', 'accessibilità'])
 list(['aspettare']) list(['migliorabile']) list(['sperare', 'expo'])
 list(['ridicolo']) list([]) list([]) list(['inconcepibile', 'pagare'])
 list(['pessimo', 'esperienza']) list(['museo', 'vecchio']) list([])
 list(['delusione']) list(['aspettare']) list(['relax'])
 list(['delusione']) list(['viaggio', 'madrid'])
 list(['navigli', 'mistero', 'targato', 'milano'])
 list(['navigare', 'percorso', 'darsena'])
 list(['servirebbe', 'modernità']) list(['reichstag']) list([])
 list(['adatto', 'bambino']) list(['servire', 'orario', 'prenotazione'])
 list(['de

In [257]:
print(titles_n[:5])

[list(['gioco', 'acqua', 'natalizio', 'navigazione', 'musicale'])
 list(['migliore']) list(['noioso']) list(['caratteristico']) list([])]


In [258]:
dictionary1=Dictionary(titles_n)
for token,id in dictionary1.token2id.items():
    print(f"{token}: {id}")

acqua: 0
gioco: 1
musicale: 2
natalizio: 3
navigazione: 4
migliore: 5
noioso: 6
caratteristico: 7
pensare: 8
sporco: 9
caos: 10
inqualificabile: 11
organizzazione: 12
berlino: 13
simbolo: 14
guidare: 15
visita: 16
godere: 17
zona: 18
interessante: 19
turistico: 20
museo: 21
prado: 22
accessibilità: 23
vergogna: 24
aspettare: 25
migliorabile: 26
expo: 27
sperare: 28
ridicolo: 29
inconcepibile: 30
pagare: 31
esperienza: 32
pessimo: 33
vecchio: 34
delusione: 35
relax: 36
madrid: 37
viaggio: 38
milano: 39
mistero: 40
navigli: 41
targato: 42
darsena: 43
navigare: 44
percorso: 45
modernità: 46
servirebbe: 47
reichstag: 48
adatto: 49
bambino: 50
orario: 51
prenotazione: 52
servire: 53
totale: 54
accogliente: 55
area: 56
ragazzino: 57
costoso: 58
iconico: 59
maestoso: 60
trascurare: 61
duomo: 62
periodo: 63
piazza: 64
sopravvalutare: 65
canale: 66
veneziano: 67
locale: 68
pieno: 69
limitato: 70
complicare: 71
opera: 72
invivibile: 73
notte: 74
sera: 75
india: 76
abbandonare: 77
degrado: 78
sto

In [275]:
#dictionary1.filter_extremes(no_above=120, keep_n=300)----> salto questo passaggio altrimenti rimangono troppe poche features

In [260]:
X4=[dictionary1.doc2bow(title) for title in titles_n]
print("Ecco i primi tre titoli in formato bow (indice,numero di occorrenze)\n ")
print(X4[:3])

Ecco i primi tre titoli in formato bow (indice,numero di occorrenze)
 
[[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1)], [(5, 1)], [(6, 1)]]


In [261]:
lda_gensim_n=LdaMulticore(X4, num_topics=3,id2word=dictionary1,passes=250,workers=3)
#1 minuto

In [262]:
for index, topic in lda_gensim_n.print_topics():
    print("\nTOPIC %d - parole più popolari" %(index+1))
    print(topic)


TOPIC 1 - parole più popolari
0.030*"milano" + 0.027*"pessimo" + 0.027*"duomo" + 0.024*"visita" + 0.021*"esperienza" + 0.021*"organizzazione" + 0.015*"museo" + 0.013*"opera" + 0.013*"migliore" + 0.013*"locale"

TOPIC 2 - parole più popolari
0.018*"sporco" + 0.014*"zona" + 0.014*"chiudere" + 0.014*"deludere" + 0.010*"mortale" + 0.010*"noia" + 0.010*"viaggio" + 0.010*"degrado" + 0.010*"valorizzare" + 0.010*"pensare"

TOPIC 3 - parole più popolari
0.052*"delusione" + 0.019*"aspettare" + 0.016*"totale" + 0.016*"caos" + 0.016*"sopravvalutare" + 0.016*"berlino" + 0.016*"sopravvalutato" + 0.016*"canale" + 0.016*"muro" + 0.012*"particolare"


In [263]:
lda_gensim_n_vis=pyLDAvis.gensim.prepare(lda_gensim_n,X4 ,dictionary1, mds='tsne')
pyLDAvis.display(lda_gensim_n_vis)

* All' interno delle parole chiave selezionate nei vari topic, se ne cominciano a vedere alcune che fanno riferimento a quli sono stati gli aspetti negativi riscontrati : caos, sporcizia, attesa, (dis)organizzazione. Queste parole sono però mischiate nei 3 topic e quindi non permettono di associare ciascun topic ad una specifica categoria di aspetti negativi. Lo stesso vale per il caso delle recensioni positive

# Conclusione

*  A conclusione del lavoro ripetiamo gli stessi passaggi dell'algoritmo di topic extraction con la libreria Gensim, utilizzando nuovamente l'intero corpus formato da tutti i commenti prodotti dagli utenti.
* Utilizziamo questa volta la codifica Tf-Idf

In [264]:
df["comment"]=corpus_cleam
df["comment_preprocessed"]=df["comment"].map(preprocess)
df.head(20)

,Unnamed: 0,comment,title,date,name,stars,comment_preprocessed
0,0,interessante sera trambusto età sacco bar rist...,Zona del canal grande,26 ottobre 2016,gfdmmad,5.0,"[interessante, sera, trambusto, età, sacco, ba..."
1,1,organizzare coda entrata piantine visita perco...,Visita di piacere,26 aprile 2019,Diabolik-Eva-014,5.0,"[organizzare, coda, entrata, piantine, visita,..."
2,2,amico o expo strada il navigli porto darsena r...,Da vedere a Milano,18 novembre 2018,O5262KGchrisd,5.0,"[amico, expo, strada, navigli, porto, darsena,..."
3,3,visitare duomo Milano madonnina guglia esperie...,Il Duomo sempre affascinante,7 agosto 2019,Babellina,5.0,"[visitare, duomo, milano, madonnina, guglia, e..."
4,4,vista duomo uscita metro togliere fiato il lav...,Eccezionale,9 dicembre 2019,Nik804,5.0,"[vista, duomo, uscita, metro, togliere, fiato,..."
5,5,Milano visitare chiesa cuore Milano passo univ...,Visita imperdibile!,21 aprile 2017,matteodifelice,5.0,"[milano, visitare, chiesa, cuore, milano, pass..."
6,6,tenere passeggiare stare,Un oasi nel caos,23 dicembre 2017,Ghert2015,4.0,"[tenere, passeggiare, stare]"
7,7,creare traccia passato impedire adeguare si es...,La Bernauerstrasse sospesa tra passato e presente,10 febbraio 2017,bobmonica,5.0,"[creare, traccia, passato, impedire, adeguare,..."
8,8,parola descrivere bellezza parco passeggiare c...,Imperdibile,6 novembre 2016,Duecuorieunaciccions,5.0,"[parola, descrivere, bellezza, parco, passeggi..."
9,9,polmone simbolo New York consigliare passeggia...,Passeggiata obbligatoria,23 novembre 2017,CDam7,5.0,"[polmone, simbolo, new, york, consigliare, pas..."


In [265]:
complete_corpus=df["comment_preprocessed"].values
print(complete_corpus)
print(complete_corpus.shape)

[list(['interessante', 'sera', 'trambusto', 'età', 'sacco', 'bar', 'ristorante', 'canal', 'street', 'negozio', 'prezzo', 'rimanere', 'soddisfatto', 'mangiare', 'zoccolo', 'portato', 'notte', 'sembrare', 'visitare', 'volere', 'perdere', 'trovare', 'milano'])
 list(['organizzare', 'coda', 'entrata', 'piantine', 'visita', 'percorso', 'controllo'])
 list(['amico', 'expo', 'strada', 'navigli', 'porto', 'darsena', 'ristrutturare', 'gioiello', 'atmosfera', 'centro', 'casa', 'negozio', 'decina', 'bar', 'ristorante', 'posizione', 'shop', 'rilassare'])
 ...
 list(['mancare', 'beato', 'mantegna', 'tiziano', 'raffaello', 'velazquez', 'goya', 'bosch', 'preferito', 'assoluto', 'fruizione', 'museo', 'permettere', 'visita'])
 list(['porta', 'simbolo', 'quadriga', 'sormonta', 'venire', 'trafugare', 'napoleone', 'bottino', 'guerra'])
 list(['darsena', 'biforcazione', 'ripristinare', 'restyling', 'fantastica', 'atmosfera', 'serata', 'locale', 'camminata', 'adiacente', 'acqua', 'fantasticare', 'frequentar

In [266]:
print(complete_corpus[:5])

[list(['interessante', 'sera', 'trambusto', 'età', 'sacco', 'bar', 'ristorante', 'canal', 'street', 'negozio', 'prezzo', 'rimanere', 'soddisfatto', 'mangiare', 'zoccolo', 'portato', 'notte', 'sembrare', 'visitare', 'volere', 'perdere', 'trovare', 'milano'])
 list(['organizzare', 'coda', 'entrata', 'piantine', 'visita', 'percorso', 'controllo'])
 list(['amico', 'expo', 'strada', 'navigli', 'porto', 'darsena', 'ristrutturare', 'gioiello', 'atmosfera', 'centro', 'casa', 'negozio', 'decina', 'bar', 'ristorante', 'posizione', 'shop', 'rilassare'])
 list(['visitare', 'duomo', 'milano', 'madonnina', 'guglia', 'esperienza', 'vivere'])
 list(['vista', 'duomo', 'uscita', 'metro', 'togliere', 'fiato', 'lavoro', 'interno', 'tralasciare'])]


In [267]:
dictionary2=Dictionary(complete_corpus)
for token,id in dictionary2.token2id.items():
    print(f"{token}: {id}")

bar: 0
canal: 1
età: 2
interessante: 3
mangiare: 4
milano: 5
negozio: 6
notte: 7
perdere: 8
portato: 9
prezzo: 10
rimanere: 11
ristorante: 12
sacco: 13
sembrare: 14
sera: 15
soddisfatto: 16
street: 17
trambusto: 18
trovare: 19
visitare: 20
volere: 21
zoccolo: 22
coda: 23
controllo: 24
entrata: 25
organizzare: 26
percorso: 27
piantine: 28
visita: 29
amico: 30
atmosfera: 31
casa: 32
centro: 33
darsena: 34
decina: 35
expo: 36
gioiello: 37
navigli: 38
porto: 39
posizione: 40
rilassare: 41
ristrutturare: 42
shop: 43
strada: 44
duomo: 45
esperienza: 46
guglia: 47
madonnina: 48
vivere: 49
fiato: 50
interno: 51
lavoro: 52
metro: 53
togliere: 54
tralasciare: 55
uscita: 56
vista: 57
affresco: 58
aspettativa: 59
bellezza: 60
chiesa: 61
cuore: 62
intento: 63
passo: 64
riuscire: 65
superare: 66
università: 67
passeggiare: 68
stare: 69
tenere: 70
adeguare: 71
creare: 72
esigenza: 73
impedire: 74
passato: 75
sito: 76
terrazzo: 77
traccia: 78
affittare: 79
colore: 80
descrivere: 81
film: 82
follia: 83

In [268]:
dictionary2.filter_extremes(no_below=5,no_above=0.6,keep_n=7500)

In [269]:
X5=[dictionary2.doc2bow(document) for document in complete_corpus]
print("Ecco i primi tre documenti in formato bow (indice,numero di occorrenze)\n ")
print(X5[:3])

Ecco i primi tre documenti in formato bow (indice,numero di occorrenze)
 
[[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1), (9, 1), (10, 1), (11, 1), (12, 1), (13, 1), (14, 1), (15, 1), (16, 1), (17, 1), (18, 1), (19, 1)], [(20, 1), (21, 1), (22, 1), (23, 1), (24, 1), (25, 1), (26, 1)], [(0, 1), (5, 1), (10, 1), (27, 1), (28, 1), (29, 1), (30, 1), (31, 1), (32, 1), (33, 1), (34, 1), (35, 1), (36, 1), (37, 1), (38, 1), (39, 1), (40, 1), (41, 1)]]


In [270]:
from gensim.models import TfidfModel

In [271]:
_tfidf_=TfidfModel(X5)
X5=_tfidf_[X5]
print("Ecco i primi tre documenti in formato tfidf \n ")
for i in range(3):
    print(X5[i])

Ecco i primi tre documenti in formato tfidf 
 
[(0, np.float64(0.18614788129313536)), (1, np.float64(0.2913459351566503)), (2, np.float64(0.2733231121057014)), (3, np.float64(0.1888652500382927)), (4, np.float64(0.0858416566832036)), (5, np.float64(0.20213304710734392)), (6, np.float64(0.20780152271022043)), (7, np.float64(0.1295415150779732)), (8, np.float64(0.1955967250521033)), (9, np.float64(0.16140458172881184)), (10, np.float64(0.1658774028103028)), (11, np.float64(0.22367636027385476)), (12, np.float64(0.2437300315714356)), (13, np.float64(0.15813430761719843)), (14, np.float64(0.405695576511709)), (15, np.float64(0.28762430033911524)), (16, np.float64(0.3782262929116536)), (17, np.float64(0.1276408441695833)), (18, np.float64(0.08054059845346961)), (19, np.float64(0.1499479821678628))]
[(20, np.float64(0.31276111373316123)), (21, np.float64(0.36497355805113135)), (22, np.float64(0.3382393276244962)), (23, np.float64(0.3420144391585053)), (24, np.float64(0.28913194669365705)), (

In [272]:
lda_gen=LdaMulticore(X5, num_topics=8, id2word=dictionary2,passes=250 ,workers=4)
#7-8 minuti

In [273]:
for index,topic in lda_gen.print_topics():
    print("\nTOPIC %d - parole più popolari" % (index+1))
    print(topic)


TOPIC 1 - parole più popolari
0.024*"parco" + 0.015*"girare" + 0.015*"bicicletta" + 0.012*"noleggiare" + 0.012*"giro" + 0.009*"bice" + 0.009*"affittare" + 0.008*"park" + 0.007*"central" + 0.007*"film"

TOPIC 2 - parole più popolari
0.009*"assoluto" + 0.007*"stancare" + 0.007*"maya" + 0.006*"illuminato" + 0.006*"settembre" + 0.005*"speciale" + 0.005*"aspettativa" + 0.005*"scontare" + 0.005*"end" + 0.005*"bus"

TOPIC 3 - parole più popolari
0.020*"locale" + 0.017*"zona" + 0.016*"milano" + 0.015*"ristorante" + 0.015*"sera" + 0.011*"bar" + 0.011*"navigli" + 0.010*"aperitivo" + 0.009*"naviglio" + 0.009*"passeggiata"

TOPIC 4 - parole più popolari
0.050*"duomo" + 0.031*"milano" + 0.029*"piazza" + 0.021*"terrazza" + 0.018*"visitare" + 0.017*"simbolo" + 0.016*"salire" + 0.015*"cattedrale" + 0.014*"vista" + 0.013*"guglia"

TOPIC 5 - parole più popolari
0.027*"chiesa" + 0.018*"affresco" + 0.017*"milano" + 0.010*"sistina" + 0.009*"duomo" + 0.009*"visitare" + 0.009*"volontario" + 0.009*"bellezza"

In [274]:
lda_gen_vis=pyLDAvis.gensim.prepare(lda_gen,X5,dictionary2,mds='tsne')
pyLDAvis.display(lda_gen_vis)